In [37]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
import sqlalchemy
import pandas as pd

In [39]:
import os
os.listdir("../../../")

['shopping_assistant.egg-info',
 '.DS_Store',
 'uv.lock',
 'ecom-backend',
 'chatbot',
 'pyproject.toml',
 'docs',
 'README.md',
 '.gitignore',
 '.env',
 '.venv',
 '.python-version',
 'docker-compose.yml',
 '.env.example',
 '.git',
 'product_retriever',
 'shopping_assistant',
 'data',
 'notebooks']

In [40]:
engine = sqlalchemy.create_engine("sqlite:///../../../ecom-backend/ecom_backend.db")

In [41]:
with engine.connect() as conn:
    df = pd.read_sql(
        """
        SELECT 
            prod.*,
            prod_h.category_name, 
            prod_h.subcategory_name,
            prod_h.category_slug, 
            prod_h.subcategory_slug
            
        FROM product prod
        JOIN product_hierarchy prod_h 
        ON (prod_h.category_id = prod.category_id)
        AND (prod_h.subcategory_id = prod.subcategory_id)
        """, 
        conn
    )

In [42]:
df = df.rename(columns={
    "id": "product_id"
})

In [43]:
df.head()

,product_id,name,slug,description,category_id,subcategory_id,price,created_at,category_name,subcategory_name,category_slug,subcategory_slug
0,1,Southwest Bracelet,southwest-bracelet,The Southwest Bracelet combines elegance with ...,4,2,169.99,2025-12-02 09:27:59,jewelry,bracelet,jewelry,bracelet
1,2,Jeweled Bracelet Watch,jeweled-bracelet-watch,"Introducing the Jeweled Bracelet Watch, a styl...",4,2,259.99,2025-12-02 09:27:59,jewelry,bracelet,jewelry,bracelet
2,3,Isis Bracelet,isis-bracelet,The Isis Bracelet add glamour to any outfit. T...,4,2,269.99,2025-12-02 09:27:59,jewelry,bracelet,jewelry,bracelet
3,4,Butterfly Mixed Metal Bracelet,butterfly-mixed-metal-bracelet,Introducing the Butterfly Mixed Metal Bracelet...,4,2,89.99,2025-12-02 09:27:59,jewelry,bracelet,jewelry,bracelet
4,5,Butterfly Gold Chain Bracelet,butterfly-gold-chain-bracelet,"Introducing the Butterfly Gold Chain Bracelet,...",4,2,59.99,2025-12-02 09:27:59,jewelry,bracelet,jewelry,bracelet


In [44]:
import weaviate
import weaviate.classes as wvc

In [45]:
import os

In [46]:
os.environ['WEAVIATE_HTTP_HOST'] = 'localhost'
os.environ['WEAVIATE_HTTP_PORT'] = '8080'
os.environ['WEAVIATE_HTTP_SECURE'] = 'false'
os.environ['WEAVIATE_GRPC_HOST'] = 'localhost'
os.environ['WEAVIATE_GRPC_PORT'] = '50051'
os.environ['WEAVIATE_GRPC_SECURE'] = 'false'

In [47]:
weaviate_connection_params = weaviate.connect.ConnectionParams(
        http={
            "host": os.getenv("WEAVIATE_HTTP_HOST"),
            "port": os.getenv("WEAVIATE_HTTP_PORT"),
            "secure": os.getenv("WEAVIATE_HTTP_SECURE")
        },
        grpc={
            "host": os.getenv("WEAVIATE_GRPC_HOST"),
            "port": os.getenv("WEAVIATE_GRPC_PORT"),
            "secure": os.getenv("WEAVIATE_GRPC_SECURE")
        }
    )

weaviate_client = weaviate.WeaviateClient(
    connection_params=weaviate_connection_params
)

In [48]:
from shopping_assistant.product_retrieval import WeaviateConnectionManager

In [49]:
VECTOR_DB_COLLECTION_NAME = "products"
OLLAMA_API_ENDPOINT = "http://ollama:11434"
EMBEDDING_MODEL = "nomic-embed-text"
PRODUCT_DATA_FIELDS = [
    "product_id",
    "name",
    "slug",
    "description",
    "price",
    "created_at",
    "category_name",
    "subcategory_name",
    "category_slug",
    "subcategory_slug",
]

WEAVIATE_INGESTION_BATCH_SIZE = 100

In [50]:
import requests
import json

response = requests.post(
    'http://localhost:11434/api/embed',
    json={
        "model": "nomic-embed-text",
        "input": "Hello, world!",
    }
)

In [34]:
with WeaviateConnectionManager(client=weaviate_client) as weaviate_client_connected:
    # products = weaviate_client_connected.collections.use(VECTOR_DB_COLLECTION_NAME)
    products = weaviate_client_connected.collections.delete('Products')
    for item in products.iterator():
        pass
        print(f"UUID: {item.uuid}")
        print(f"Properties: {item.properties}")
        break
    # You can access specific properties like:
    # print(item.properties["your_property_name"])

AttributeError: 'NoneType' object has no attribute 'iterator'

In [33]:
item.properties

{'product_id': 164,
 'description': "The Velvet Heels are a luxurious blend of comfort and elegance, designed to elevate any outfit with their timeless style. These heeled shoes feature a sleek, closed toe design with a luxurious velvet upper, providing a touch of opulence and sophistication. The 3-inch heel height offers a balanced and stylish lift, while the cushioned sole ensures all-day comfort. The open-back design makes them easy to slip on and off, making them perfect for both casual and formal occasions. The vibrant red color adds a pop of cheer to any ensemble, whether you're dressing up for a festive party or keeping it simple for a casual night out.",
 'price': 139.99,
 'category': 'footwear',
 'subcategory': 'shoes',
 'name': 'Velvet Heels'}

In [30]:
item.properties

{'product_id': 164,
 'description': "The Velvet Heels are a luxurious blend of comfort and elegance, designed to elevate any outfit with their timeless style. These heeled shoes feature a sleek, closed toe design with a luxurious velvet upper, providing a touch of opulence and sophistication. The 3-inch heel height offers a balanced and stylish lift, while the cushioned sole ensures all-day comfort. The open-back design makes them easy to slip on and off, making them perfect for both casual and formal occasions. The vibrant red color adds a pop of cheer to any ensemble, whether you're dressing up for a festive party or keeping it simple for a casual night out.",
 'price': 139.99,
 'category': 'footwear',
 'subcategory': 'shoes',
 'name': 'Velvet Heels'}

In [51]:
with WeaviateConnectionManager(client=weaviate_client) as weaviate_client_connected:
    weaviate_client_connected.collections.delete(VECTOR_DB_COLLECTION_NAME)

    products = weaviate_client_connected.collections.create(
        name=VECTOR_DB_COLLECTION_NAME,
        vector_config=wvc.config.Configure.Vectors.text2vec_ollama(
            api_endpoint=OLLAMA_API_ENDPOINT,
            model=EMBEDDING_MODEL,
            source_properties=["name", "description", "category_name", "subcategory_name"]
        ),
        properties=[
            wvc.config.Property(
                name="product_id",
                data_type=wvc.config.DataType.INT
            )
        ]
    )

    with products.batch.fixed_size(batch_size=WEAVIATE_INGESTION_BATCH_SIZE) as batch:
        for idx, row in df[PRODUCT_DATA_FIELDS].iterrows():
            rec = row.to_dict()
            batch.add_object(properties=rec)

{'message': 'Failed to send all objects in a batch of 100', 'error': 'WeaviateInsertManyAllFailedError(\'Every object failed during insertion. Here is the set of all errors: send POST request: Post "http://ollama:11434/api/embed": dial tcp: lookup ollama on 127.0.0.11:53: no such host\')'}
{'message': 'Failed to send 100 objects in a batch of 100. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send all objects in a batch of 100', 'error': 'WeaviateInsertManyAllFailedError(\'Every object failed during insertion. Here is the set of all errors: send POST request: Post "http://ollama:11434/api/embed": dial tcp: lookup ollama on 127.0.0.11:53: no such host\')'}
{'message': 'Failed to send 100 objects in a batch of 100. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send all objects in a batch of 18', 'error': 'WeaviateInsertManyAllFa

In [52]:
from shopping_assistant.product_retrieval import retrieve_products

In [53]:
responses = retrieve_products(
    categories=["jewelry"],
    weaviate_client=weaviate_client,
    n=2000
)

In [54]:
set(obj.properties["category_slug"] for obj in responses.objects)

{'jewelry'}

In [56]:
obj.properties.keys()

dict_keys(['description', 'category_slug', 'price', 'subcategory_name', 'created_at', 'product_id', 'slug', 'category_name', 'subcategory_slug', 'name'])

In [55]:
for obj in responses.objects:
    print(json.dumps(obj.properties, indent=2))

{
  "category_slug": "jewelry",
  "description": "The Snowdrop Floral Earrings are a charming blend of natural beauty and modern design, perfect for adding a touch of elegance to your summer wardrobe. These ivory snowdrop earrings are made from a lightweight and durable polamer that bring a touch of nature's charm, making them ideal for both casual outings and more formal events. Pair with a floral dress or jeans for a look that's both chic and comfortable.",
  "product_id": 8,
  "price": 59.99,
  "created_at": "2025-12-02 09:27:59",
  "subcategory_name": "earrings",
  "slug": "snowdrop-floral-earrings",
  "category_name": "jewelry",
  "subcategory_slug": "earrings",
  "name": "Snowdrop Floral Earrings"
}
{
  "description": "Introducing the Butterfly Mixed Metal Bracelet, a perfect blend of modern and chic sophistication. This bracelet is designed to add a touch of flair to your look, making it a versatile accessory for any occasion. The bracelet is crafted from bronze and metal with b